# Taxi Trip Fetching

In this notebook we fetch the taxi trip data provided by the city of chicago: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data (downloaded on 05.05.2026).

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd

import requests

# weather data
import openmeteo_requests
import requests_cache
from retry_requests import retry

import time

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch trip data info                    #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

URL = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
params = {
    "$select": "count(*) AS n",
    "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' "
              "AND trip_start_timestamp < '2026-01-01T00:00:00'",
}
print(requests.get(URL, params=params, timeout=120).json())

[{'n': '6825838'}]


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch trip data                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import io

API_ENDPOINT = "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
LIMIT = 50000
MAX_RETRIES = 5
OUTPUT_FILE = "CHICAGO_TAXI_TRIPS_2025.csv"

def fetch_page(last_ts, last_id):
    # Keyset pagination: jump directly to where we left off using an index-friendly filter.
    # Avoids the O(n) row-skip cost of offset-based pagination.
    if last_ts is None:
        where = (
            "trip_start_timestamp >= '2025-01-01T00:00:00' "
            "AND trip_start_timestamp < '2026-01-01T00:00:00'"
        )
    else:
        where = (
            f"trip_start_timestamp < '2026-01-01T00:00:00' "
            f"AND (trip_start_timestamp > '{last_ts}' "
            f"OR (trip_start_timestamp = '{last_ts}' AND trip_id > '{last_id}'))"
        )
    params = {
        "$where": where,
        "$order": "trip_start_timestamp, trip_id",
        "$limit": LIMIT,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(API_ENDPOINT, params=params, timeout=120, stream=True) as r:
                r.raise_for_status()
                return r.content
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

last_ts, last_id = None, None
first_page = True
total_rows = 0

with open(OUTPUT_FILE, "wb") as f:
    while True:
        print(f"Fetching next page (cursor: {last_ts})...")
        content = fetch_page(last_ts, last_id)

        df = pd.read_csv(io.BytesIO(content))
        row_count = len(df)
        print(f"  Got {row_count} rows. Total so far: {total_rows + row_count}")

        if row_count == 0:
            break

        if first_page:
            df.to_csv(f, index=False)
            first_page = False
        else:
            df.to_csv(f, index=False, header=False)

        total_rows += row_count
        last_ts = df["trip_start_timestamp"].iloc[-1]
        last_id = df["trip_id"].iloc[-1]

        if row_count < LIMIT:
            break

print(f"Download complete. {total_rows} rows written to {OUTPUT_FILE}.")

Fetching next page (cursor: None)...
  Got 50000 rows. Total so far: 50000
Fetching next page (cursor: 2025-01-05T15:30:00.000)...
  Attempt 1/5 failed: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)
  Attempt 2/5 failed: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)
  Got 50000 rows. Total so far: 100000
Fetching next page (cursor: 2025-01-09T08:00:00.000)...
  Got 50000 rows. Total so far: 150000
Fetching next page (cursor: 2025-01-13T08:45:00.000)...
  Got 50000 rows. Total so far: 200000
Fetching next page (cursor: 2025-01-16T10:00:00.000)...
  Attempt 1/5 failed: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)
  Attempt 2/5 failed: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)
  Attempt 3/5 failed: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read ti

In [7]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Check data                              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

trip_data = pd.read_csv("CHICAGO_TAXI_TRIPS_2025.csv")
trip_data.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location
0,0275e2d8147a31e1ce320c5fb15f9910563cafe1,84957c8960b674346784746bbc1d48cafff4976b162323...,2025-01-01T00:00:00.000,2025-01-01T00:15:00.000,758.0,2.93,NaN,NaN,8.0,8.0,...,0.0,9.75,Prcard,Flash Cab,41.899602,-87.633308,POINT (-87.6333080367 41.899602111),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
1,05aa05bf6f3ec476715fad9f706bd137e08e00b7,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,2025-01-01T00:00:00.000,2025-01-01T00:15:00.000,1233.0,13.66,NaN,NaN,76.0,6.0,...,5.0,48.60,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),41.944227,-87.655998,POINT (-87.6559981815 41.9442266014)
2,060f692c4ca69e552c07a97c99715ad51112470e,e533bfdc483206f9c02c1c879a118d88f0a3ca1cd2703f...,2025-01-01T00:00:00.000,2025-01-01T00:00:00.000,6.0,0.00,NaN,NaN,6.0,6.0,...,0.0,30.50,Credit Card,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.944227,-87.655998,POINT (-87.6559981815 41.9442266014)
3,0b2d1368456148b049af41dc8aa565d688081756,2d5029b701200a2ab66bd5071f743efe20a3036597d2b3...,2025-01-01T00:00:00.000,2025-01-01T00:00:00.000,11.0,0.00,NaN,NaN,77.0,77.0,...,0.0,25.62,Credit Card,Sun Taxi,41.986712,-87.663416,POINT (-87.6634164054 41.9867117999),41.986712,-87.663416,POINT (-87.6634164054 41.9867117999)
4,17365c83264f028a307ac70308d770fe03bcbcae,c3f8e0b6712bf3ea80e75ddde065b0ed42aa530e8c40cb...,2025-01-01T00:00:00.000,2025-01-01T00:15:00.000,985.0,3.22,NaN,NaN,8.0,7.0,...,0.0,55.00,Cash,City Service,41.899602,-87.633308,POINT (-87.6333080367 41.899602111),41.922686,-87.649489,POINT (-87.6494887289 41.9226862843)


In [8]:
trip_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6825838 entries, 0 to 6825837
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   trip_id                     str    
 1   taxi_id                     str    
 2   trip_start_timestamp        str    
 3   trip_end_timestamp          str    
 4   trip_seconds                float64
 5   trip_miles                  float64
 6   pickup_census_tract         float64
 7   dropoff_census_tract        float64
 8   pickup_community_area       float64
 9   dropoff_community_area      float64
 10  fare                        float64
 11  tips                        float64
 12  tolls                       float64
 13  extras                      float64
 14  trip_total                  float64
 15  payment_type                str    
 16  company                     str    
 17  pickup_centroid_latitude    float64
 18  pickup_centroid_longitude   float64
 19  pickup_centroid_location    str 

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch weather data					  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.8781,
	"longitude": 87.6298,
	"start_date": "2024-01-01",
	"end_date": "2026-01-01",
	"daily": ["sunrise", "sunset", "daylight_duration", "sunshine_duration"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_speed_100m", "is_day", "sunshine_duration", "direct_radiation"],
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(5).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_speed_100m = hourly.Variables(10).ValuesAsNumpy()
hourly_is_day = hourly.Variables(11).ValuesAsNumpy()
hourly_sunshine_duration = hourly.Variables(12).ValuesAsNumpy()
hourly_direct_radiation = hourly.Variables(13).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
hourly_data["is_day"] = hourly_is_day
hourly_data["sunshine_duration"] = hourly_sunshine_duration
hourly_data["direct_radiation"] = hourly_direct_radiation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()
daily_daylight_duration = daily.Variables(2).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)
